<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline Final con Análisis Multicriterio
**Maestría en Ingeniería - Universidad EAFIT** 
**Proyecto:** Detección de deslizamientos mediante IA

Este notebook integra el entrenamiento de Random Forest con métricas avanzadas: Importancia de Variables, Matriz de Confusión y Curva ROC, corregido para NumPy 2.0.

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Rutas de datos
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir = img_dirs[0]
    train_mask_dir = train_img_dir.parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ {len(img_list)} muestras detectadas.")
else:
    print("❌ ERROR: Verifica la ruta en Drive.")

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Extrayendo características"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # --- CORRECCIÓN NUMPY 2.0 ---
    rgb = patch[:,:,[3,2,1]]
    rgb_norm = ((rgb - np.min(rgb)) / (np.ptp(rgb) + 1e-8) * 255).astype("uint8")
    
    # Descriptores: HOG + Slope (Banda 13)
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), channel_axis=-1, feature_vector=True)
    avg_slope = np.mean(patch[:,:,12])
    
    X.append(np.hstack([feats_hog, avg_slope]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset listo: {X.shape}")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

print("Entrenando Baseline...")
for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[t_idx], y[t_idx])
    
    preds = rf.predict(X[v_idx])
    f1 = f1_score(y[v_idx], preds)
    f1_scores.append(f1)
    print(f"Fold {fold+1}: F1 = {f1:.4f}")

print(f"\n🚀 F1-SCORE FINAL: {np.mean(f1_scores):.4f}")

# --- VISUALIZACIONES ---
fig, ax = plt.subplots(1, 3, figsize=(22, 6))

# 1. Importancia de Variables
importances = rf.feature_importances_
indices = np.argsort(importances)[-15:]
ax[0].barh(range(len(indices)), importances[indices], color='teal')
ax[0].set_yticks(range(len(indices)))
ax[0].set_yticklabels([f'HOG_{i}' if i < X.shape[1]-1 else 'PENDIENTE (B13)' for i in indices])
ax[0].set_title("Top 15 Importancia de Características")

# 2. Matriz de Confusión
cm = confusion_matrix(y[v_idx], preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Estable', 'Desliz.'])
disp.plot(cmap='YlGnBu', ax=ax[1], colorbar=False)
ax[1].set_title("Matriz de Confusión (Último Fold)")

# 3. Curva ROC
y_probs = rf.predict_proba(X[v_idx])[:, 1]
fpr, tpr, _ = roc_curve(y[v_idx], y_probs)
roc_auc = auc(fpr, tpr)
ax[2].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
ax[2].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
ax[2].set_title("Curva ROC")
ax[2].legend(loc="lower right")

plt.tight_layout()
plt.show()